# Laboratory: Pandas in Practice: Data Assemblies, Cleaning, and Modeling Prep
**Course:** Mathematical Foundations of Machine and Deep Learning

**Duration:** 90 minutes (1.5h)

**Environment:** Google Colab / Local Jupyter Notebook

## 1. Plan & Objectives

### Lab Objectives
1.  Understand the concept of **Data Assembly**—merging normalized database tables into a single feature matrix ($X$).
2.  Identify and fix common data quality issues (duplicates, bad data types, outliers).
3.  Perform complex data aggregations using `groupby` and `pivot_table`.
4.  Correctly split data and handle missing values **without data leakage**.

### After this lab, you can...
*   [x] Parse messy string columns into usable numeric/datetime formats.
*   [x] Visualize missingness and distributions to spot outliers.
*   [x] Combine multiple tables using `.merge()` and `.pivot_table()`.
*   [x] Prepare a clean $(X, y)$ dataset and split it safely.

## 2. Intro
**Typical Data Quality Problems to highlight:**
1.  **Format errors:** Strings instead of floats (`"$15.00"` $\rightarrow$ `15.0`).
2.  **Outliers/Errors:** A user aged `999` years.
3.  **Missing Data:** Nulls (`NaN`).
4.  **Duplicates:** A transaction logged twice due to a network error.
5.  **Leakage:** Using information from the test set to clean the training set (e.g., calculating the global mean to fill missing values before splitting).


## 3 Exercises

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# Plot styling
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (8, 5)

In [ ]:
# --- SYNTHETIC DATA GENERATION (Do not modify) ---
np.random.seed(42)

In [ ]:
# 1. Customers Table
n_cust = 200
customer_ids = np.arange(1, n_cust + 1)
ages = np.random.normal(35, 10, n_cust)
ages[np.random.choice(n_cust, 15, replace=False)] = np.nan # Introduce NaNs
ages[np.random.choice(n_cust, 5, replace=False)] = 999     # Introduce Outliers

customers = pd.DataFrame({
    'customer_id': customer_ids,
    'age': ages,
    'signup_date': [f"2023-{np.random.randint(1,13):02d}-{np.random.randint(1,28):02d}" for _ in range(n_cust)],
    'churn': np.random.choice([0, 1], n_cust, p=[0.8, 0.2]) # Target variable
})

In [ ]:
# 2. Products Table
products = pd.DataFrame({
    'product_id': ['P1', 'P2', 'P3'],
    'category': ['Electronics', 'Clothing', 'Food']
})

In [ ]:
# 3. Orders Table
n_orders = 500
orders = pd.DataFrame({
    'order_id': np.arange(1001, 1001 + n_orders),
    'customer_id': np.random.choice(customer_ids, n_orders),
    'product_id': np.random.choice(['P1', 'P2', 'P3'], n_orders),
    # Introduce bad formatting (string with $)
    'amount': [f"${np.random.uniform(10, 150):.2f}" for _ in range(n_orders)]
})
# Introduce duplicates
orders = pd.concat([orders, orders.sample(15, random_state=42)])
# Shuffle
orders = orders.sample(frac=1, random_state=42).reset_index(drop=True)

print("Data generated successfully!")

### Task 1: Initial Quality Checks & Visualizing Issues
**Description:** Before cleaning, we must understand the mess. Visualize the missing values in `customers` and the distribution of `age` to spot outliers. Also, check for duplicates in `orders`.


In [ ]:
# TODO: 1. Count duplicated rows in the 'orders' dataframe
num_duplicates = None # YOUR CODE HERE
print(f"Number of duplicate orders: {num_duplicates}")

In [ ]:
# TODO: 2. Plot the distribution (histogram) of the 'age' column in 'customers'.
# Notice the extreme values!
# YOUR CODE HERE (use sns.histplot or plt.hist)

In [ ]:
# TODO: 3. Create a bar chart showing the count of missing values per column in 'customers'
# Hint: use .isna().sum()
# YOUR CODE HERE

In [ ]:
# --- CHECK ---
assert num_duplicates == 15, "Did you count exact duplicates correctly?"
print("Task 1 Complete! Look at the plots to understand the data issues.")

### Task 2: Handling Duplicates & Bad Data Types
**Description:** Machine learning models need numbers, not strings. Remove the duplicate rows in `orders`. Then, clean the `amount` column by removing the `$` sign and converting it to a `float`. Convert `signup_date` to datetime.


In [ ]:
# TODO: 1. Drop duplicate rows from 'orders' (modify the dataframe in place or reassign)
# YOUR CODE HERE


In [ ]:
# TODO: 2. Clean 'amount' column: remove '$' and cast to float
orders['amount'] = None # YOUR CODE

In [ ]:
# TODO: 3. Convert 'signup_date' in customers to pandas datetime objects
customers['signup_date'] = None # YOUR CODE HERE

In [ ]:
# --- CHECK ---
assert orders.duplicated().sum() == 0, "There are still duplicates!"
assert pd.api.types.is_float_dtype(orders['amount']), "'amount' is not a float!"
assert pd.api.types.is_datetime64_any_dtype(customers['signup_date']), "Date not converted!"
print("Task 2 Complete!")

### Task 3: Handling Outliers (Preparation)
**Description:** We saw that some customers have an age of `999`. This is clearly a data entry error. Replace any age $> 100$ with `np.nan`.
*Important:* Do **not** impute (fill) the NaNs yet! We will do that after the train/test split to avoid data leakage.

In [ ]:
# TODO: Replace ages strictly greater than 100 with np.nan
# Hint: You can use .loc or np.where
# YOUR CODE HERE

In [ ]:
# --- CHECK ---
assert customers['age'].max() < 100, "There are still outliers in age!"
print(f"Current missing ages count: {customers['age'].isna().sum()}")
print("Task 3 Complete!")

### Task 4: Data Assembly - Merging Tables
**Description:** We need to combine `orders` with `products` so we know the category of each purchase. Use a `.merge()`.

In [ ]:
# TODO: Merge 'orders' and 'products' on 'product_id'. Use a left join.
orders_detailed = None # YOUR CODE HERE

In [ ]:
assert 'category' in orders_detailed.columns, "Merge failed! 'category' column missing."
assert len(orders_detailed) == 500, "Merge altered the number of rows incorrectly!"
print("Task 4 Complete!\n", orders_detailed.head(3))

### Task 5: Groupby & Aggregation (Feature Engineering)
**Description:** The `orders_detailed` table has multiple rows per customer. Our ML model expects **one row per customer**. Group by `customer_id` and calculate:
1. Total amount spent.
2. Number of orders placed.

In [ ]:
# TODO: Group by customer_id and calculate total sum of 'amount' and count of 'order_id'
# Hint: use .groupby('customer_id').agg({'amount': 'sum', 'order_id': 'count'})
customer_aggs = None # YOUR CODE HERE

In [ ]:
# Rename columns for clarity
customer_aggs = customer_aggs.rename(columns={'amount': 'total_spent', 'order_id': 'order_count'})

In [ ]:
# --- CHECK ---
assert 'total_spent' in customer_aggs.columns, "Missing total_spent column!"
assert customer_aggs.index.name == 'customer_id', "Index should be customer_id"
print("Task 5 Complete!\n", customer_aggs.head(3)

### Task 6: Pivot Tables
**Description:** Let's extract more granular features. How much did each customer spend in *each category*? Create a pivot table from `orders_detailed`. Fill missing values with `0` (if a customer didn't buy Electronics, they spent 0 on it).


In [ ]:
# TODO: Create a pivot table
# index: 'customer_id', columns: 'category', values: 'amount', aggfunc: 'sum', fill_value: 0
category_spend = None # YOUR CODE HERE

In [ ]:
# --- CHECK ---
assert 'Electronics' in category_spend.columns, "Pivot table missing categories!"
assert category_spend.isna().sum().sum() == 0, "There should be no NaNs in the pivot table (use fill_value=0)"
print("Task 6 Complete!\n", category_spend.head(3))

### Task 7: Final Assembly, Train/Test Split, and Leakage-Free Imputation
**Description:**
1. Merge `customers`, `customer_aggs`, and `category_spend` into a final `df_model`.
2. Define $X$ (features) and $y$ (target: `churn`).
3. Split the data ($80\%$ train, $20\%$ test).
4. **Leakage-free Imputation:** Calculate the median `age` using **ONLY** $X_{train}$. Use this median to fill missing values in both $X_{train}$ and $X_{test}$.


In [ ]:
# 1. Final Assembly (Left join customers with aggregations)
df_model = customers.merge(customer_aggs, on='customer_id', how='left')
df_model = df_model.merge(category_spend, on='customer_id', how='left')

In [ ]:
# Fill NaN for customers who made NO orders (total_spent = 0, order_count = 0)
features_to_fill = ['total_spent', 'order_count', 'Electronics', 'Clothing', 'Food']
df_model[features_to_fill] = df_model[features_to_fill].fillna(0)

In [ ]:
# Drop columns not useful for modeling (IDs, Dates)
df_model = df_model.drop(columns=['customer_id', 'signup_date'])

In [ ]:
# 2. Define X and y
# TODO: y is 'churn', X is everything else
X = None # YOUR CODE HERE
y = None # YOUR CODE HERE

In [ ]:
# 3. Train Test Split
# TODO: Split X and y (test_size=0.2, random_state=42)
X_train, X_test, y_train, y_test = None # YOUR CODE HERE

In [ ]:
# 4. Leakage-Free Imputation
# TODO: Calculate median age from X_train ONLY
train_median_age = None # YOUR CODE HERE

In [ ]:
# TODO: Fill NaNs in X_train and X_test using this median
# YOUR CODE HERE

In [ ]:
# --- CHECK ---
assert X_train['age'].isna().sum() == 0, "Missing values remain in X_train!"
assert X_test['age'].isna().sum() == 0, "Missing values remain in X_test!"
assert 'churn' not in X_train.columns, "Target variable 'churn' leaked into features!"
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print("Task 7 Complete! The dataset is ready for Machine Learning.")

## Summary

### Conclusions
*   **Data Assembly is crucial:** ML algorithms need a 2D matrix (rows = samples, columns = features). Building this from relational databases requires carefully planned `.merge()`, `.groupby()`, and `.pivot_table()` operations.
*   **Formatting matters:** Hidden characters (like `$`) or wrong data types will crash your models. Always check `.dtypes`.
*   **Avoid Data Leakage at all costs:** Never use information from your entire dataset (like a global mean/median) to fill missing values *before* splitting into train/test. Always split first, fit the imputer on train, and transform train and test.

### Checklist: "Before Modeling" (10 Points)
*   [ ] 1. Do I have a single flat table (Feature Matrix $X$) and a separate Target ($y$)?
*   [ ] 2. Are all my features numeric? (No stray strings, dates parsed into intervals/days).
*   [ ] 3. Have I removed exact duplicate rows?
*   [ ] 4. Are the data types correct? (`int` or `float` for math, `category` for categorical).
*   [ ] 5. Have I checked for extreme outliers (e.g., age = 999) and handled them?
*   [ ] 6. Did I handle missing values (`NaN`) appropriately (drop, impute, or flag)?
*   [ ] 7. **Crucial:** Did I split the data into Train and Test *before* applying global transformations?
*   [ ] 8. Did I ensure the target variable ($y$) is completely removed from $X$? (No target leakage).
*   [ ] 9. Did I handle customers/users who had *no activity* properly? (e.g., filling NaNs with `0` after a left join).
*   [ ] 10. Do my features make logical sense? (Sanity check via `.describe()`).

### 3 Control Questions (Ask the audience)
1.  **Question:** Why do we convert strings like `"$150"` to floats before modeling?
2.  **Question:** What is the difference between `.groupby()` and `.pivot_table()` in our feature engineering?
3.  **Question:** Why is it considered a mistake to calculate the median age of the *entire* dataset and use it to replace missing values before using `train_test_split`?